# [P1] Q2 E: scientific interpretation (report)


## [P1] How this report is built: text generated from the results

**Decision:** the paragraphs of this report that contain numbers are **written by code**, not typed
as fixed text. Each such cell takes the numbers directly from the analysis (Q2 C and D), puts them
into sentences, and displays the result as formatted text (`IPython.display.Markdown`).

**Why not simply a text cell?**
* **The numbers can never go out of date.** The analysis of Q2 C and D may still change before
  submission (a cleaning step, a threshold, a group). With typed numbers, the text would silently
  disagree with the figures and tables; here, re-running the notebook updates every number.
  This follows decision `g_verify`: every number in the report is produced by code.
* **The wording also follows the results.** Some sentences are judgements, not numbers: for example
  "there is a clear premium" holds only if the whole confidence interval is above 1. Such words are
  chosen by a small rule in the code (`verdict` functions below), so the conclusion changes if the
  result changes.
* **Claims that the text relies on are checked.** Where a sentence depends on a fact that a rule
  cannot rephrase (for example "removing the near-duplicate rows hardly changes the result"),
  an `assert` stops the notebook if the fact no longer holds, so someone must re-read that paragraph.

**Cost:** the report is visible only after the notebook has run. This matches decision `g_out`
(the final notebooks are run from scratch and saved with their outputs).

## [P1] Setup: load the results of Q2 C and D
*Temporary:* `%run` executes the analysis notebook of Part 3, so all its tables and variables are
available here. At integration this report is placed after Q2 D in `Q2_laptop_analysis.ipynb`,
and this cell is deleted.

In [ ]:
%run Q2_part3_analysis.ipynb

In [ ]:
from IPython.display import Markdown, Image, display

def pct(ratio):
    """A price ratio as a percent difference, e.g. 1.33 -> '33 %'."""
    return f"{(ratio - 1) * 100:.0f} %"

def interval_verdict(low, high):
    """Wording for a ratio with a 95 % interval: is there a clear difference from 1 (no difference)?"""
    if low > 1:
        return "a clear premium (the whole interval is above 1)"
    if high < 1:
        return "a clear discount (the whole interval is below 1)"
    return "no clear premium (the interval includes 1)"

def show_figure(file_name, width=750):
    """Display a saved figure from the figures folder."""
    display(Image(filename=str(FIG_DIR / file_name), width=width))

# small checks of the helpers on known inputs
assert pct(1.334) == "33 %" and pct(2.0) == "100 %"
assert interval_verdict(1.06, 1.24).startswith("a clear premium")
assert interval_verdict(0.95, 1.11).startswith("no clear premium")
assert interval_verdict(0.80, 0.95).startswith("a clear discount")

## [P1] Q2 E: Analytical report on the laptop dataset

In [ ]:
# ---- numbers used in the summary ----
n_laptops = len(df)
ram_raw = ram_price_adjusted.loc["0 RAM only"]
ram_adj = ram_price_adjusted.loc["5 + brand"]
apple_acer = brand_adjusted.loc["Apple vs Acer"]
msi_lenovo = brand_adjusted.loc["MSI vs Lenovo"]

summary = f"""
### Summary
We examined {n_laptops:,} laptops to see which characteristics go with a higher **catalogue price**.
Laptops with more RAM, an SSD, or a premium brand cost much more, but most of these raw gaps are
shared with the rest of the specification. Each doubling of RAM goes with a price
{ram_raw['pct_change']:.0f} % higher in the raw data, but only {ram_adj['pct_change']:.0f} % higher
among laptops with a similar CPU tier, type, storage, resolution and brand.
SSD-only laptops cost {ssd_vs_hdd['median ratio SSD/HDD']:.1f} times as much as HDD-only laptops
(medians). After controlling for the specification, Apple keeps a premium of about
{pct(apple_acer['adjusted_ratio'])} over Acer, while MSI against Lenovo shows
{interval_verdict(msi_lenovo['adj_ci_low'], msi_lenovo['adj_ci_high'])}.
Unusual prices exist but look like real machines, and removing them hardly changes these results.
All findings are associations in this dataset, not causes.
"""
display(Markdown(summary))

### Finding 1: RAM and price

In [ ]:
raw = ram_price_raw
steps = ram_price_adjusted["pct_change"]
drops = -steps.diff().dropna()                       # how much each added group of controls lowers the RAM effect
biggest_drop = drops.idxmax().split("+ ")[-1]        # e.g. "CPU tier"
ram_no_dups = sensitivity.loc["Q1 RAM: % price change per doubling (model 5)", "without near-duplicates"]

# claims that the text relies on
assert ram_adj["ci_low_pct"] > 0, "the adjusted RAM effect is no longer clearly positive: re-read this finding"
assert abs(ram_no_dups - ram_adj["pct_change"]) < 5, "near-duplicates now change the RAM effect: re-read"

finding_ram = f"""
| | |
|---|---|
| **What we observed** | RAM and price rise together: Spearman {raw['Spearman (same on raw and log scale)']:.2f}, Pearson {raw['Pearson, raw (GB vs euro)']:.2f} (raw) and {raw['Pearson, log-log']:.2f} (log-log). The median price grows from {groups.loc['4', 'median_price']:,.0f} euro at 4 GB to {groups.loc['16', 'median_price']:,.0f} euro at 16 GB. Without controls, each doubling of RAM goes with a price {ram_raw['pct_change']:.0f} % higher (95 % interval {ram_raw['ci_low_pct']:.0f} to {ram_raw['ci_high_pct']:.0f} %). |
| **What the analysis supports** | RAM is still associated with price among laptops with a similar specification: after controlling for CPU tier, laptop type, SSD, resolution and brand, a doubling of RAM goes with a price {ram_adj['pct_change']:.0f} % higher ({ram_adj['ci_low_pct']:.0f} to {ram_adj['ci_high_pct']:.0f} %). This is much smaller than the raw effect; the largest drop comes from adding the **{biggest_drop}**. The result is almost the same without the near-duplicate rows ({ram_no_dups:.0f} %). |
| **What we cannot conclude** | That adding RAM to a laptop raises its price by {ram_adj['pct_change']:.0f} %. This is a comparison between different laptops, not an experiment; characteristics we did not measure (processor generation, graphics card, build quality) may travel with RAM. The model assumes the same percentage step for every doubling, while the median step is not constant ({ratio_4_to_8:.1f}× from 4 to 8 GB, {ratio_8_to_16:.1f}× from 8 to 16 GB). |
"""
display(Markdown(finding_ram))
show_figure("q2_ram_vs_price_scatter.png")

### Finding 2: SSD against HDD (`Storage_type`)

In [ ]:
s = ssd_vs_hdd
ssd_no_dups = sensitivity.loc["Q2 SSD/HDD median ratio", "without near-duplicates"]

assert s["median ratio, 95% CI low"] > 1, "SSD-only laptops are no longer clearly more expensive: re-read"
assert abs(ssd_no_dups - s["median ratio SSD/HDD"]) < 0.2, "near-duplicates now change the SSD result: re-read"

finding_ssd = f"""
| | |
|---|---|
| **What we observed** | SSD-only laptops (n = {s['n SSD-only']:.0f}) have a median price of {s['median price SSD-only (euro)']:,.0f} euro, against {s['median price HDD-only (euro)']:,.0f} euro for HDD-only laptops (n = {s['n HDD-only']:.0f}): a ratio of {s['median ratio SSD/HDD']:.2f} (95 % interval {s['median ratio, 95% CI low']:.2f} to {s['median ratio, 95% CI high']:.2f}). |
| **What the analysis supports** | The gap is large and not driven by a few extreme prices: in {s['P(random SSD laptop costs more than random HDD laptop)'] * 100:.0f} % of pairs (one SSD-only and one HDD-only laptop) the SSD laptop is the more expensive (rank-biserial {s['rank-biserial (SSD vs HDD)']:.2f}). Without the near-duplicate rows the ratio is {ssd_no_dups:.2f}. |
| **What we cannot conclude** | That an SSD makes a laptop {s['median ratio SSD/HDD']:.1f} times as expensive. SSDs are more common in newer and stronger laptops, and this comparison does not control for the rest of the specification. Laptops with both drives ({storage_table.loc['SSD + HDD', 'n']:.0f}) and with Flash Storage ({storage_table.loc['Flash Storage', 'n']:.0f}) were shown but not tested. |
"""
display(Markdown(finding_ssd))
show_figure("q2_ssd_vs_hdd_boxplot.png")

### Finding 3: Brand premium

In [ ]:
rows = []
for pair, r in brand_adjusted.iterrows():
    rows.append(f"| {pair} | {r['raw_median_ratio']:.2f} | {r['adjusted_ratio']:.2f} "
                f"({r['adj_ci_low']:.2f} to {r['adj_ci_high']:.2f}) | {interval_verdict(r['adj_ci_low'], r['adj_ci_high'])} |")
brand_rows = "\n".join(rows)

n_apple = brand_table.loc["Apple", "n"]
with_premium = [p for p, r in brand_adjusted.iterrows() if r["adj_ci_low"] > 1]
without_premium = [p for p in brand_adjusted.index if p not in with_premium]
max_change = sensitivity.loc[[m for m in sensitivity.index if "adjusted ratio" in m], "difference"].abs().max()

assert (brand_adjusted["adjusted_ratio"] < brand_adjusted["raw_median_ratio"]).all(), \
    "an adjusted brand gap is no longer smaller than the raw gap: re-read"
assert max_change < 0.05, "near-duplicates now change the brand results: re-read"

finding_brand = f"""
| Pair | Raw ratio of medians | Ratio after controls (95 % interval) | Verdict after controls |
|---|---|---|---|
{brand_rows}

| | |
|---|---|
| **What we observed** | Raw medians: Apple {brand_table.loc['Apple', 'median']:,.0f}, MSI {brand_table.loc['MSI', 'median']:,.0f}, Lenovo {brand_table.loc['Lenovo', 'median']:,.0f} and Acer {brand_table.loc['Acer', 'median']:,.0f} euro. All four raw gaps are large and significant after Holm correction (largest corrected p-value {brand_raw['p_holm'].max():.1e}). |
| **What the analysis supports** | Most of the raw gap is explained by the specification (RAM, SSD, resolution, CPU tier, type). A premium remains for {', '.join(with_premium) if with_premium else 'no pair'}; {', '.join(without_premium) if without_premium else 'no pair'} shows no clear premium. Removing the near-duplicate rows changes no adjusted ratio by more than {max_change:.3f}. |
| **What we cannot conclude** | That the brand name itself causes the premium: the graphics card, processor generation, screen and build quality are not controlled and may differ between brands. Apple has only {n_apple:.0f} laptops, so its intervals are wide. The result also depends on the reference brand. |
"""
display(Markdown(finding_brand))
show_figure("q2_brand_premium.png")

### Finding 4: Unusual observations

In [ ]:
n_tukey = int(df["outlier_tukey_log"].sum())
n_resid = int(df["outlier_resid"].sum())
n_weight = int(df["outlier_weight"].sum())
n_resid_high = int((df["outlier_resid"] & (std_resid > 0)).sum())
tukey_rows = df.loc[df["outlier_tukey_log"], ["Company", "Product", "Price (Euro)"]]
tukey_text = ", ".join(f"{r.Company} {r.Product} ({r._3:,.0f} euro)" for r in tukey_rows.itertuples())

ba = before_after
all_col, both_col = "all laptops", [c for c in ba.columns if c.startswith("without both")][0]
spearman_change = abs(ba.loc["Spearman", both_col] - ba.loc["Spearman", all_col])
mean_change = ba.loc["mean price (euro)", all_col] - ba.loc["mean price (euro)", both_col]
median_change = ba.loc["median price (euro)", all_col] - ba.loc["median price (euro)", both_col]
ram_effect_range = ba.loc["RAM effect, % per doubling"]

assert spearman_change < 0.02, "removing outliers now changes Spearman noticeably: re-read"
assert mean_change >= median_change, "the mean is no longer more sensitive than the median: re-read"

finding_outliers = f"""
| | |
|---|---|
| **What we observed** | Tukey fences on the log price ({np.exp(fence_low):,.0f} to {np.exp(fence_high):,.0f} euro) flag {n_tukey} laptop: {tukey_text}. Comparing each price with what the specification predicts flags {n_resid} laptops ({n_resid_high} more expensive than expected); the weight check flags {n_weight}, mostly heavy 17-inch gaming laptops. |
| **What the analysis supports** | Almost all unusual observations look like real machines (gaming laptops, workstations, business laptops), not errors, so we **kept** them. Removing the Tukey and residual outliers lowers the mean price by {mean_change:.0f} euro but the median by only {median_change:.0f} euro, changes Spearman by {"less than 0.001" if spearman_change < 0.001 else f"{spearman_change:.3f}"}, and the RAM effect stays between {ram_effect_range.min():.0f} and {ram_effect_range.max():.0f} % per doubling across all scenarios. The conclusions above do not depend on them. |
| **What we cannot conclude** | That a flagged laptop is an error, or that an unflagged one is correct: we can only check internal consistency, not the real market. The flags depend on our thresholds (1.5 × IQR, standardized residual above 3). |
"""
display(Markdown(finding_outliers))

In [ ]:
n_suspicious = int(df["suspicious"].sum())
n_near_dup = int(df["near_duplicate"].sum())
small_ram = ram_levels[ram_levels["fewer_than_20"]]
small_ram_text = ", ".join(f"{level} GB ({n:.0f})" for level, n in small_ram["n"].items())

limitations = f"""
### Limitations
1. **Confounding.** CPU tier, laptop type and brand affect both RAM and price, so raw associations are
   inflated. We report results before and after controlling for them and make no causal claim.
   Characteristics we did not measure (processor generation, graphics card model, build quality) remain uncontrolled.
2. **Near-duplicate rows are not independent.** {n_near_dup} rows are identical except for the price.
   We kept them, so p-values are optimistic; every key result was recomputed without them and barely changes.
3. **Unknown collection date.** The dataset has no documentation of when prices were collected, so the
   results describe this dataset, not today's market.
4. **Catalogue prices.** The prices are listed prices, not prices actually paid.
5. **Small groups give unstable estimates.** Apple has {n_apple:.0f} laptops; RAM levels with fewer than 20 laptops:
   {small_ram_text}. We state group sizes, compare only groups of at least 20, and report intervals.

Further notes on how the variables were built:
* **Internal checks only.** We cannot verify that a model exists in the real market. Five plausibility
  rules (for example a small screen but a heavy weight) flagged {n_suspicious} {"row" if n_suspicious == 1 else "rows"}; flagged rows were
  examined, not removed.
* **`Dedicated_GPU`** is decided by the graphics company (Nvidia or AMD), but some AMD chips are integrated graphics.
* **`CPU_tier`** has a mixed `Other` group (Celeron, Pentium, Atom, AMD, Core M): it separates strong from
  weak processors, not fine differences.
* **Storage size:** 1 TB is counted as 1000 GB.
* **Representativeness:** we do not know whether the dataset is a random sample of the laptop market.
"""
display(Markdown(limitations))

In [ ]:
figures_used = ["q2_ram_vs_price_scatter.png", "q2_ram_vs_price_boxplot.png",
                "q2_ssd_vs_hdd_boxplot.png", "q2_brand_premium.png"]
missing = [f for f in figures_used if not (FIG_DIR / f).exists()]
assert not missing, f"figures missing: {missing}"

display(Markdown("### Figures cited\n" + "\n".join(f"* `figures/{f}`" for f in figures_used)))

In [ ]:
readme_summary = f"""**Q2 – laptop prices.** In {n_laptops:,} laptops, more RAM, an SSD and a premium brand all go with
higher catalogue prices, but much of each raw gap is shared with the rest of the specification:
a doubling of RAM goes with +{ram_raw['pct_change']:.0f} % raw but +{ram_adj['pct_change']:.0f} % after controls,
and Apple keeps about +{(apple_acer['adjusted_ratio'] - 1) * 100:.0f} % over Acer. These are associations, not causes."""
print(readme_summary)